# Task 2 – MongoDB Atlas and Python

This notebook connects securely to MongoDB Atlas, migrates `pokemon_data.csv` with pandas and PyMongo, demonstrates MongoDB's flexible document schema, and performs fundamental CRUD operations.

## 1. Import libraries

The connection string is read from the `MONGODB_URI` environment variable. Credentials are not stored in this notebook.

In [1]:
import os
from pathlib import Path

import pandas as pd
from pymongo import MongoClient, ReplaceOne
from pymongo.server_api import ServerApi
from pymongo.errors import PyMongoError
from dotenv import load_dotenv

## 2. Connect to MongoDB Atlas

A ping command verifies that Python can communicate with the Atlas cluster. The task uses the `bde` database and `pokemon` collection.

In [2]:
load_dotenv()
uri = os.environ.get("MONGODB_URI")
if not uri:
    raise RuntimeError("Set MONGODB_URI before running this notebook.")

client = MongoClient(
    uri,
    server_api=ServerApi("1"),
    serverSelectionTimeoutMS=10_000,
)

try:
    ping_result = client.admin.command("ping")
    print("Connected successfully. Ping status:", ping_result["ok"])
except PyMongoError as error:
    client.close()
    raise ConnectionError(f"Unable to connect to MongoDB Atlas: {error}") from error

database = client["bde"]
collection = database["pokemon"]

Connected successfully. Ping status: 1


## 3. Load and prepare the CSV dataset

pandas reads the structured CSV file. Converting the DataFrame to `object` before replacing missing values ensures pandas `NaN` values become Python `None`, which MongoDB stores as `null`.

In [3]:
csv_path = Path("pokemon_data.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Dataset not found: {csv_path.resolve()}")

df = pd.read_csv(csv_path)
df = df.astype(object).where(pd.notnull(df), None)

print("Dataset shape:", df.shape)
display(df[["pokedex_number", "name", "type1", "type2"]].head())

Dataset shape: (801, 41)


,pokedex_number,name,type1,type2
0,1,Bulbasaur,grass,poison
1,2,Ivysaur,grass,poison
2,3,Venusaur,grass,poison
3,4,Charmander,fire,None
4,5,Charmeleon,fire,None


## 4. Migrate the CSV records to MongoDB

`pokedex_number` is indexed as a unique business identifier. Bulk upserts insert new Pokémon and replace existing matching Pokémon, preventing duplicate copies when this cell is rerun.

In [4]:
collection.create_index("pokedex_number", unique=True, sparse=True)

records = df.to_dict(orient="records")
for record in records:
    record["record_source"] = "pokemon_data.csv"

operations = [
    ReplaceOne(
        {"pokedex_number": record["pokedex_number"]},
        record,
        upsert=True,
    )
    for record in records
]

if operations:
    import_result = collection.bulk_write(operations, ordered=False)
    print("New Pokémon inserted:", import_result.upserted_count)
    print("Existing Pokémon matched:", import_result.matched_count)

csv_document_count = collection.count_documents(
    {"record_source": "pokemon_data.csv"}
)
print("CSV documents stored in Atlas:", csv_document_count)

New Pokémon inserted: 801
Existing Pokémon matched: 0
CSV documents stored in Atlas: 801


## 5. Insert flexible-schema documents

These documents deliberately use different fields and data types. MongoDB accepts them in the same collection without requiring every document to follow a fixed table schema. Upserts make the demonstration repeatable.

In [5]:
flexible_documents = [
    {
        "demo_id": "FLEX-001",
        "name": "DataMon",
        "type1": "digital",
        "abilities": ["Streaming", "Analytics"],
    },
    {
        "demo_id": "FLEX-002",
        "name": "CloudMon",
        "region": "Atlas",
        "is_experimental": True,
        "metrics": {"latency_ms": 12, "availability": 99.9},
    },
    {
        "demo_id": "FLEX-DELETE",
        "name": "TemporaryMon",
        "temporary": True,
    },
]

for document in flexible_documents:
    collection.replace_one(
        {"demo_id": document["demo_id"]},
        document,
        upsert=True,
    )

print("Flexible documents available:")
for document in collection.find(
    {"demo_id": {"$in": ["FLEX-001", "FLEX-002", "FLEX-DELETE"]}},
    {"_id": 0},
).sort("demo_id", 1):
    print(document)

Flexible documents available:
{'demo_id': 'FLEX-001', 'name': 'DataMon', 'type1': 'digital', 'abilities': ['Streaming', 'Analytics']}
{'demo_id': 'FLEX-002', 'name': 'CloudMon', 'region': 'Atlas', 'is_experimental': True, 'metrics': {'latency_ms': 12, 'availability': 99.9}}
{'demo_id': 'FLEX-DELETE', 'name': 'TemporaryMon', 'temporary': True}


## 6. Find and filter documents

This query retrieves Grass-type Pokémon with an attack score of at least 80. The result is limited to five documents to keep the evidence concise.

In [6]:
find_query = {"type1": "grass", "attack": {"$gte": 80}}

for document in collection.find(find_query).limit(5):
    print(
        document["name"],
        "- attack:", document["attack"],
        "- generation:", document["generation"],
    )

Venusaur - attack: 100 - generation: 1
Vileplume - attack: 80 - generation: 1
Weepinbell - attack: 90 - generation: 1
Victreebel - attack: 105 - generation: 1
Exeggutor - attack: 105 - generation: 1


## 7. Projection

Projection returns only selected attributes and excludes MongoDB's internal `_id` field.

In [7]:
projection = {
    "_id": 0,
    "name": 1,
    "type1": 1,
    "attack": 1,
    "generation": 1,
}

for document in collection.find({"type1": "fire"}, projection).limit(5):
    print(document)

{'attack': 52, 'name': 'Charmander', 'type1': 'fire', 'generation': 1}
{'attack': 64, 'name': 'Charmeleon', 'type1': 'fire', 'generation': 1}
{'attack': 104, 'name': 'Charizard', 'type1': 'fire', 'generation': 1}
{'attack': 41, 'name': 'Vulpix', 'type1': 'fire', 'generation': 1}
{'attack': 67, 'name': 'Ninetales', 'type1': 'fire', 'generation': 1}


## 8. Update a document

The update targets a demonstration document, preserving the original CSV records. Matched and modified counts verify the operation.

In [8]:
update_result = collection.update_one(
    {"demo_id": "FLEX-001"},
    {"$set": {"attack": 55, "reviewed": True}},
)

print("Matched:", update_result.matched_count)
print("Modified:", update_result.modified_count)
print(collection.find_one({"demo_id": "FLEX-001"}, {"_id": 0}))

Matched: 1
Modified: 1
{'demo_id': 'FLEX-001', 'name': 'DataMon', 'type1': 'digital', 'abilities': ['Streaming', 'Analytics'], 'attack': 55, 'reviewed': True}


## 9. Delete a document

Only the temporary demonstration document is deleted. No source Pokémon record is removed.

In [9]:
delete_result = collection.delete_one({"demo_id": "FLEX-DELETE"})
print("Deleted:", delete_result.deleted_count)
print(
    "Temporary document still exists:",
    collection.count_documents({"demo_id": "FLEX-DELETE"}) > 0,
)

Deleted: 1
Temporary document still exists: False


## 10. Final verification

The final counts confirm that the migrated dataset and the two retained flexible-schema documents are present in Atlas.

In [10]:
verification = {
    "csv_documents": collection.count_documents(
        {"record_source": "pokemon_data.csv"}
    ),
    "flexible_documents": collection.count_documents(
        {"demo_id": {"$in": ["FLEX-001", "FLEX-002"]}}
    ),
    "grass_pokemon": collection.count_documents({"type1": "grass"}),
    "legendary_pokemon": collection.count_documents({"is_legendary": 1}),
}

print(verification)

{'csv_documents': 801, 'flexible_documents': 2, 'grass_pokemon': 78, 'legendary_pokemon': 70}


## 11. Close the connection

Closing the client releases network resources after all database operations are complete.

In [11]:
client.close()
print("MongoDB connection closed.")

MongoDB connection closed.


## Conclusion

This task established a secure Python connection to MongoDB Atlas, migrated structured CSV data into a document collection, demonstrated flexible schemas, and completed the requested find, projection, update, and delete operations.